### 🧠 Grouped Query Attention with Causal Sliding Window (PyTorch) — Full Step-by-Step Explanation

This code implements **Grouped Query Attention (GQA)** combined with a **causal sliding window mask**.
It is a **memory-efficient attention mechanism** used in many modern LLMs like **LLaMA 2** and **Mistral**.

It improves two major problems of classic **Multi‑Head Attention**:

* 🔹 **Too much KV cache memory**
* 🔹 **Too many tokens attending to everything**

This implementation solves both by combining:

1️⃣ **Grouped Query Attention (GQA)** → reduces KV memory
2️⃣ **Sliding Window Attention** → reduces attention computation

---

### 🧩 Step 1 — Imports

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
```

### Purpose

| Library             | Purpose                |
| ------------------- | ---------------------- |
| torch               | tensor operations      |
| torch.nn            | neural network layers  |
| torch.nn.functional | activation + softmax   |
| math                | mathematical functions |

---

### 🧠 Step 2 — Define the Attention Class

```python
class GQACausalSlidingAttention(nn.Module):
```

This creates a **custom attention layer** inside a neural network.

All attention logic will be implemented inside this class.

---

### ⚙️ Step 3 — Initialization Function

```python
def __init__(self, embed_dim, num_heads, num_kv_heads, window_size):
```

### Parameters

| Parameter    | Meaning                   |
| ------------ | ------------------------- |
| embed_dim    | embedding dimension       |
| num_heads    | number of query heads     |
| num_kv_heads | number of key/value heads |
| window_size  | sliding window size       |

---

### 📏 Step 4 — Important Attention Dimensions

```python
self.head_dim = embed_dim // num_heads
```

This calculates the **dimension of each attention head**.

### Formula

$$
head_dim = \frac{embed_dim}{num_heads}
$$

### Example

If

```
embed_dim = 4096
num_heads = 32
```

then

$$
head_dim = 4096 / 32 = 128
$$

---

### Why split embeddings?

Instead of one huge attention space:

```
4096 dim
```

we split into **multiple smaller attention spaces**

```
32 heads × 128 dim
```

Each head learns **different relationships**.

Example heads:

| Head | Learns              |
| ---- | ------------------- |
| 1    | syntax              |
| 2    | semantic similarity |
| 3    | long dependencies   |
| 4    | entities            |

---

### 👥 Step 5 — Grouped Query Attention

```python
self.group_size = num_heads // num_kv_heads
```

### Formula

$$
group_size = \frac{num_heads}{num_kv_heads}
$$

### Example

```
num_heads = 32
num_kv_heads = 8
```

Then

$$
group_size = 32 / 8 = 4
$$

Meaning:

```
4 query heads share the same K and V
```

Visualization:

```
Query Heads:   Q1 Q2 Q3 Q4 | Q5 Q6 Q7 Q8 | ...
KV Heads:      K1 V1       | K2 V2
```

This **reduces KV memory by 4×**.

---

### 🔧 Step 6 — Linear Projections

```python
self.q_proj = nn.Linear(embed_dim, embed_dim)
self.k_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim)
self.v_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim)
```

These layers convert the input embeddings into:

* Queries
* Keys
* Values

---

### Query Projection

```python
self.q_proj = nn.Linear(embed_dim, embed_dim)
```

Formula:

$$
Q = X W_Q
$$

Where

| Symbol | Meaning             |
| ------ | ------------------- |
| X      | input embedding     |
| WQ     | query weight matrix |

---

### Key Projection

```python
self.k_proj = nn.Linear(embed_dim, num_kv_heads * head_dim)
```

Formula

$$
K = X W_K
$$

But only for **KV heads** (smaller than query heads).

---

### Value Projection

```python
self.v_proj = nn.Linear(embed_dim, num_kv_heads * head_dim)
```

Formula

$$
V = X W_V
$$

---

### 🔄 Step 7 — Output Projection

```python
self.out_proj = nn.Linear(embed_dim, embed_dim)
```

After attention, we project back.

Formula

$$
Output = Attention(Q,K,V) W_O
$$

---

### 🧱 Step 8 — Sliding Causal Mask Function

```python
def create_sliding_causal_mask(self, T, device):
```

Creates a **mask matrix**.

---

### Mask Initialization

```python
mask = torch.full((T, T), float("-inf"), device=device)
```

Creates matrix:

```
T × T
```

Filled with

```
-∞
```

Why?

Because in softmax:

$$
e^{-\infty} = 0
$$

So masked tokens get **zero attention**.

---

### 🔁 Step 9 — Sliding Window Logic

```python
for i in range(T):
```

Loop over tokens.

Example tokens:

```
T0 T1 T2 T3 T4 T5
```

---

### Window Start

```python
start = max(0, i - window_size + 1)
```

Example

```
window_size = 3
```

For token **T5**

```
start = 5 - 3 + 1 = 3
```

So visible tokens:

```
T3 T4 T5
```

---

### Allow attention

```python
mask[i, start:i+1] = 0
```

Example mask

```
      T0 T1 T2 T3 T4 T5
T0    0
T1    0 0
T2    0 0 0
T3      0 0 0
T4        0 0 0
T5          0 0 0
```

This enforces:

* **causality**
* **local attention**

---

### 🚀 Step 10 — Forward Pass

```python
def forward(self, x):
```

Input tensor:

```
x shape
```

```
(B, T, C)
```

---

### 📦 What are B, T, C?

| Symbol | Meaning                  |
| ------ | ------------------------ |
| B      | Batch size               |
| T      | Sequence length (tokens) |
| C      | Embedding dimension      |

Example:

```
B = 4
T = 1024
C = 4096
```

Tensor shape:

```
(4, 1024, 4096)
```

Meaning:

```
4 sequences
1024 tokens each
4096 embedding dimension
```

---

### 🔎 Step 11 — Query Projection

```python
q = self.q_proj(x)
```

Shape

```
(B,T,C)
```

---

### Reshape

```python
q = q.view(B, T, self.num_heads, self.head_dim)
```

Shape becomes

```
(B, T, H, D)
```

Where

| Symbol | Meaning         |
| ------ | --------------- |
| H      | number of heads |
| D      | head_dim        |

Formula

$$
C = H \times D
$$

---

### Transpose

```python
q = q.transpose(1, 2)
```

Shape

```
(B, H, T, D)
```

Why?

Because attention operates **per head**.

---

### 🔑 Step 12 — Key Projection

```python
k = self.k_proj(x)
```

Shape

```
(B,T,num_kv_heads*D)
```

Reshape

```
(B,T,num_kv_heads,D)
```

Transpose

```
(B,num_kv_heads,T,D)
```

---

### 💎 Step 13 — Value Projection

Same logic as keys.

Final shape

```
(B,num_kv_heads,T,D)
```

---

### 🔁 Step 14 — Expand KV for Groups

```python
k = k.repeat_interleave(self.group_size, dim=1)
v = v.repeat_interleave(self.group_size, dim=1)
```

Example

```
num_heads = 32
num_kv_heads = 8
group_size = 4
```

Before expansion

```
K shape = (B,8,T,D)
```

After expansion

```
(B,32,T,D)
```

Now queries and KV match.

---

### 📊 Step 15 — Attention Scores

```python
scores = torch.matmul(q, k.transpose(-2, -1))
```

Matrix multiplication.

Formula

$$
Attention = QK^T
$$

Shape

```
(B,H,T,T)
```

Meaning

```
each token attends to every token
```

---

### ⚖️ Step 16 — Scaling

```python
scores = scores / math.sqrt(self.head_dim)
```

Formula

$$
Attention = \frac{QK^T}{\sqrt{d_k}}
$$

Why?

Without scaling:

```
large dot products
```

Softmax becomes unstable.

---

### 🪟 Step 17 — Apply Sliding Mask

```python
mask = self.create_sliding_causal_mask(T, x.device)
scores = scores + mask
```

This removes attention outside window.

---

### 🔥 Step 18 — Softmax

```python
attn = F.softmax(scores, dim=-1)
```

Formula

$$
softmax(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}
$$

This converts scores into **probabilities**.

---

### 📊 Step 19 — Weighted Sum

```python
out = torch.matmul(attn, v)
```

Formula

$$
Output = softmax(QK^T)V
$$

Shape

```
(B,H,T,D)
```

---

### 🔄 Step 20 — Merge Heads

```python
out = out.transpose(1, 2).contiguous().view(B, T, C)
```

Before merge

```
(B,H,T,D)
```

After merge

```
(B,T,C)
```

Formula

$$
C = H \times D
$$

---

### 🎯 Step 21 — Final Projection

```python
return self.out_proj(out)
```

Final output

```
(B,T,C)
```

Same dimension as input.

---

### 🚀 Why This Attention is Powerful

### 1️⃣ Memory Reduction (GQA)

KV memory reduction

$$
Reduction = \frac{num_heads}{num_kv_heads}
$$

Example

```
32 → 8
```

Memory reduced **4×**.

---

### 2️⃣ Computation Reduction (Sliding Window)

Normal attention complexity

$$
O(T^2)
$$

Sliding window

$$
O(T \times window)
$$

Example

```
T = 8192
window = 512
```

Huge savings.

---

### ⚠️ Limitations

| Issue                      | Reason                           |
| -------------------------- | -------------------------------- |
| long context understanding | sliding window limits visibility |
| KV replication             | small overhead                   |
| less global info           | tokens outside window ignored    |

---

✅ **Summary**

This code combines:

| Technique      | Purpose                   |
| -------------- | ------------------------- |
| GQA            | reduce KV memory          |
| Sliding window | reduce attention compute  |
| Causal masking | autoregressive generation |

This design is used in modern efficient transformers like:

* Mistral
* LLaMA 2

---

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class GQACausalSlidingAttention(nn.Module):
    def __init__(
        self,
        embed_dim,
        num_heads,
        num_kv_heads,
        window_size
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.window_size = window_size

        self.head_dim = embed_dim // num_heads
        self.group_size = num_heads // num_kv_heads

        # projections
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def create_sliding_causal_mask(self, T, device):

        mask = torch.full((T, T), float("-inf"), device=device)

        for i in range(T):

            start = max(0, i - self.window_size + 1)

            mask[i, start:i+1] = 0

        return mask

    def forward(self, x):

        B, T, C = x.shape

        # ---- Q projection ----
        q = self.q_proj(x)
        q = q.view(B, T, self.num_heads, self.head_dim)
        q = q.transpose(1, 2)

        # ---- K projection ----
        k = self.k_proj(x)
        k = k.view(B, T, self.num_kv_heads, self.head_dim)
        k = k.transpose(1, 2)

        # ---- V projection ----
        v = self.v_proj(x)
        v = v.view(B, T, self.num_kv_heads, self.head_dim)
        v = v.transpose(1, 2)

        # ---- expand KV for query groups ----
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)

        # ---- attention scores ----
        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        # ---- sliding causal mask ----
        mask = self.create_sliding_causal_mask(T, x.device)

        scores = scores + mask

        # ---- softmax ----
        attn = F.softmax(scores, dim=-1)

        # ---- weighted sum ----
        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)

### 🧠 Visual Diagram of Tensor Reshaping in Attention (Step-by-Step)

Understanding tensor reshaping in attention becomes much easier if you **visualize how the tensor changes shape at every step**. We'll follow the exact flow used in your **Grouped Query Attention with sliding window** implementation.

---

### 1️⃣ Input Tensor

The model receives token embeddings from the transformer block.

Shape:

$$
X \in \mathbb{R}^{B \times T \times C}
$$

| Symbol | Meaning                  |
| ------ | ------------------------ |
| **B**  | batch size               |
| **T**  | sequence length (tokens) |
| **C**  | embedding dimension      |

Example:

```
B = 2
T = 6
C = 8
```

Visualization:

```
Input X

Batch 1
T0 → [e1 e2 e3 e4 e5 e6 e7 e8]
T1 → [e1 e2 e3 e4 e5 e6 e7 e8]
T2 → [e1 e2 e3 e4 e5 e6 e7 e8]
T3 → [e1 e2 e3 e4 e5 e6 e7 e8]
T4 → [e1 e2 e3 e4 e5 e6 e7 e8]
T5 → [e1 e2 e3 e4 e5 e6 e7 e8]
```

Shape:

```
(B,T,C) = (2,6,8)
```

---

### 2️⃣ Linear Projection (Q,K,V)

The embedding is projected into **query, key, value spaces**.

Formula:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

After projection:

```
Q shape = (B,T,C)
K shape = (B,T,KV_heads × head_dim)
V shape = (B,T,KV_heads × head_dim)
```

Example:

```
num_heads = 4
num_kv_heads = 2
head_dim = 2
```

So:

```
C = 4 × 2 = 8
```

---

### 3️⃣ Reshaping Query Into Heads

Code:

```python
q = q.view(B, T, num_heads, head_dim)
```

Formula:

$$
C = H \times D
$$

Where

| Symbol | Meaning         |
| ------ | --------------- |
| H      | number of heads |
| D      | head dimension  |

Example reshape:

```
(2,6,8)
↓
(2,6,4,2)
```

Visualization:

```
Token embedding

[ e1 e2 | e3 e4 | e5 e6 | e7 e8 ]
   H1      H2      H3      H4
```

Each **pair of numbers becomes one attention head**.

---

### 4️⃣ Transpose for Attention Computation

Code:

```python
q = q.transpose(1,2)
```

Shape becomes:

```
(B, H, T, D)
```

Example:

```
(2,4,6,2)
```

Visualization:

```
Head 1
T0 → [e1 e2]
T1 → [e1 e2]

Head 2
T0 → [e3 e4]
T1 → [e3 e4]
```

Each head now processes tokens **independently**.

---

### 5️⃣ Key and Value Reshaping

Keys and values use **fewer heads in GQA**.

Example:

```
num_heads = 4
num_kv_heads = 2
```

Code:

```python
k = k.view(B,T,num_kv_heads,head_dim)
```

Shape:

```
(2,6,4)
↓
(2,6,2,2)
```

After transpose:

```
(2,2,6,2)
```

Visualization:

```
KV Heads

Head1
T0 → [k1 k2]

Head2
T0 → [k3 k4]
```

---

### 6️⃣ KV Expansion (Grouped Query Attention)

Now we expand KV heads so queries can use them.

Code:

```python
k = k.repeat_interleave(group_size, dim=1)
```

Example:

```
num_heads = 4
num_kv_heads = 2
group_size = 2
```

Before:

```
(B,2,T,D)
```

After:

```
(B,4,T,D)
```

Mapping:

```
Query Heads      KV Head

Q1 Q2 → KV1
Q3 Q4 → KV2
```

Visualization:

```
Q1 → K1
Q2 → K1
Q3 → K2
Q4 → K2
```

---

### 7️⃣ Attention Score Matrix

Dot product:

$$
Attention = QK^T
$$

Shape:

```
(B,H,T,T)
```

Example:

```
(2,4,6,6)
```

Visualization:

```
Token Attention Matrix

      T0 T1 T2 T3 T4 T5
T0    ■
T1    ■ ■
T2    ■ ■ ■
T3    ■ ■ ■ ■
T4    ■ ■ ■ ■ ■
T5    ■ ■ ■ ■ ■ ■
```

With sliding window only local tokens are visible.

---

### 8️⃣ Multiply With Values

Formula:

$$
Output = softmax(QK^T)V
$$

Shape:

```
(B,H,T,D)
```

Example:

```
(2,4,6,2)
```

---

### 9️⃣ Merge Heads Back

Code:

```python
out = out.transpose(1,2).view(B,T,C)
```

Before:

```
(B,H,T,D)
```

After:

```
(B,T,C)
```

Example:

```
(2,6,8)
```

Visualization:

```
Heads merged

[H1 | H2 | H3 | H4]
↓
[ e1 e2 e3 e4 e5 e6 e7 e8 ]
```

---

### 🔁 Full Tensor Flow

```
Input
(B,T,C)

↓ linear projection

Q → (B,T,C)
K → (B,T,KV*D)
V → (B,T,KV*D)

↓ reshape

Q → (B,T,H,D)
K → (B,T,KV,D)
V → (B,T,KV,D)

↓ transpose

Q → (B,H,T,D)
K → (B,KV,T,D)
V → (B,KV,T,D)

↓ KV expand

K,V → (B,H,T,D)

↓ attention

scores → (B,H,T,T)

↓ output

(B,H,T,D)

↓ merge heads

(B,T,C)
```

---

### 🚀 Why This Design Is Important

This tensor manipulation allows transformers to:

1️⃣ **parallelize attention heads**
2️⃣ **learn multiple relationships simultaneously**
3️⃣ **reduce KV cache memory using GQA**

---

✅ **Key intuition**

```
Embedding → split into heads
Heads → compute attention independently
Heads → merge back into embedding
```

---

If you'd like, I can also show something **extremely useful for LLM engineers**:

### 🔥 The **exact tensor shapes used inside models like LLaMA, Mistral, and GPT-4 for GQA + KV cache during inference** (this is where most people get confused).
